# Geração de Máscaras TIFF com Modificação

Este notebook replica o comportamento de criação automática de máscaras TIFF usando os parâmetros de `args.yaml`.

**Características:**
- Carrega todas as orthoimages das 3 regiões com todos os seus canais
- Gera máscaras seguindo o mesmo processo do código principal
- Permite modificação das máscaras antes de salvá-las


## 1. Imports e Configuração

In [ ]:
import sys
from pathlib import Path
import numpy as np
import rasterio
from scipy.ndimage import binary_fill_holes
from skimage.morphology import convex_hull_image
from PIL import Image
import matplotlib.pyplot as plt

# Adicionar o diretório raiz ao path
PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))

from src.io_operations import (
    read_yaml, 
    normalize_multi_region_args,
    fix_relative_paths,
    read_tiff,
    get_image_metadata,
    array2raster
)
from src.utils import check_folder

print(f"PROJECT_ROOT: {PROJECT_ROOT}")

## 2. Carregar Parâmetros do args.yaml

In [ ]:
# Carregar args.yaml
args_path = PROJECT_ROOT / "args.yaml"
args = read_yaml(str(args_path))

# Normalizar paths relativos para absolutos
fix_relative_paths(args)

# Normalizar configuração multi-região
args = normalize_multi_region_args(args)

# Parâmetros para geração de máscara (padrões do código)
make_convex = True
fill_holes = True
save_preview = True
preview_max_size = 1024

print(f"Número de regiões: {args.get('num_regions', len(args.get('ortho_images', [])))}")
print(f"Orthoimages:")
for i, ortho_path in enumerate(args.get('ortho_images', [])):
    print(f"  Região {i}: {ortho_path}")

## 3. Carregar Todas as Orthoimages das 3 Regiões

In [ ]:
# Carregar todas as orthoimages com todos os canais
orthoimages = {}
orthoimages_metadata = {}

num_regions = args.get('num_regions', len(args.get('ortho_images', [])))

for region_idx in range(num_regions):
    ortho_path = args['ortho_images'][region_idx]
    
    print(f"\nCarregando região {region_idx}: {Path(ortho_path).name}")
    
    # Carregar orthoimage com todos os canais
    ortho = read_tiff(ortho_path)
    
    # Obter metadados
    metadata = get_image_metadata(ortho_path)
    
    orthoimages[region_idx] = ortho
    orthoimages_metadata[region_idx] = metadata
    
    print(f"  Shape: {ortho.shape}")
    print(f"  Dtype: {ortho.dtype}")
    print(f"  Min: {ortho.min()}, Max: {ortho.max()}")
    if ortho.ndim == 3:
        print(f"  Canais: {ortho.shape[0]}")
    else:
        print(f"  Imagem monocromática")
    print(f"  CRS: {metadata.get('crs')}")
    print(f"  Transform: {metadata.get('transform')}")

## 4. Gerar Máscaras para Cada Região

In [ ]:
# Gerar máscaras iniciais seguindo o mesmo processo do código
masks = {}

for region_idx in range(num_regions):
    ortho = orthoimages[region_idx]
    
    print(f"\nGerando máscara inicial para região {region_idx}...")
    
    # Handle different shapes: (bands, height, width) or (height, width)
    if ortho.ndim == 3:
        # Multiband image: pixel is non-empty if ANY channel is non-zero
        mask = np.any(ortho != 0, axis=0)
        print(f"  Multiband image com shape {ortho.shape}")
    else:
        # Single band image
        mask = ortho != 0
        print(f"  Single band image com shape {ortho.shape}")
    
    # Count initial valid pixels
    initial_valid = np.sum(mask)
    total_pixels = mask.size
    print(f"  Pixels válidos iniciais: {initial_valid:,} / {total_pixels:,} ({100*initial_valid/total_pixels:.2f}%)")
    
    # Fill holes in the mask
    if fill_holes:
        mask = binary_fill_holes(mask)
        print(f"  Após fill_holes: {np.sum(mask):,} pixels válidos")
    
    # Make the mask convex
    if make_convex:
        mask = convex_hull_image(mask)
        print(f"  Após convex_hull: {np.sum(mask):,} pixels válidos")
    
    # Convert to uint8 (0 and 1)
    mask = mask.astype(np.uint8)
    
    masks[region_idx] = mask
    
    print(f"  Máscara final shape: {mask.shape}, dtype: {mask.dtype}")

## 5. Visualização das Máscaras Antes da Modificação

In [ ]:
# Visualizar máscaras geradas
fig, axes = plt.subplots(1, num_regions, figsize=(6*num_regions, 6))
if num_regions == 1:
    axes = [axes]

for region_idx in range(num_regions):
    mask = masks[region_idx]
    
    # Redimensionar para visualização se muito grande
    max_size = 2000
    if mask.shape[0] > max_size or mask.shape[1] > max_size:
        scale = min(max_size / mask.shape[0], max_size / mask.shape[1])
        new_h = int(mask.shape[0] * scale)
        new_w = int(mask.shape[1] * scale)
        from PIL import Image
        mask_vis = Image.fromarray((mask * 255).astype(np.uint8), mode='L')
        mask_vis = mask_vis.resize((new_w, new_h), Image.Resampling.NEAREST)
        mask_vis = np.array(mask_vis) / 255.0
    else:
        mask_vis = mask
    
    axes[region_idx].imshow(mask_vis, cmap='gray')
    axes[region_idx].set_title(f"Região {region_idx} - Máscara Inicial\n{Path(args['ortho_images'][region_idx]).name}")
    axes[region_idx].axis('off')

plt.tight_layout()
plt.show()

## 6. Modificação das Máscaras

**ESPAÇO PARA SUAS MODIFICAÇÕES**

Aqui você pode modificar as máscaras antes de salvá-las. As máscaras estão disponíveis no dicionário `masks`.

Exemplo de modificação:
```python
# masks[0] = sua_modificacao(masks[0])
# masks[1] = sua_modificacao(masks[1])
# masks[2] = sua_modificacao(masks[2])
```

In [ ]:
# ============================================
# ESPAÇO PARA SUAS MODIFICAÇÕES
# ============================================
# Modifique as máscaras aqui antes de salvá-las
# As máscaras estão em masks[0], masks[1], masks[2]

# Exemplo (descomente e modifique conforme necessário):
# from scipy import ndimage
# 
# for region_idx in range(num_regions):
#     mask = masks[region_idx]
#     
#     # Exemplo: aplicar erosão
#     # mask = ndimage.binary_erosion(mask, structure=np.ones((5,5))).astype(np.uint8)
#     
#     # Exemplo: aplicar dilatação
#     # mask = ndimage.binary_dilation(mask, structure=np.ones((5,5))).astype(np.uint8)
#     
#     # Exemplo: aplicar opening
#     # mask = ndimage.binary_opening(mask, structure=np.ones((3,3))).astype(np.uint8)
#     
#     # Exemplo: aplicar closing
#     # mask = ndimage.binary_closing(mask, structure=np.ones((3,3))).astype(np.uint8)
#     
#     # Exemplo: remover pequenos objetos
#     # from skimage.morphology import remove_small_objects
#     # mask = remove_small_objects(mask.astype(bool), min_size=1000).astype(np.uint8)
#     
#     masks[region_idx] = mask

print("Máscaras prontas para modificação. Adicione seu código acima.")

## 7. Visualização das Máscaras Após Modificação (se aplicável)

In [ ]:
# Visualizar máscaras após modificação (opcional)
# Descomente se quiser visualizar após suas modificações

# fig, axes = plt.subplots(1, num_regions, figsize=(6*num_regions, 6))
# if num_regions == 1:
#     axes = [axes]
# 
# for region_idx in range(num_regions):
#     mask = masks[region_idx]
#     
#     # Redimensionar para visualização se muito grande
#     max_size = 2000
#     if mask.shape[0] > max_size or mask.shape[1] > max_size:
#         scale = min(max_size / mask.shape[0], max_size / mask.shape[1])
#         new_h = int(mask.shape[0] * scale)
#         new_w = int(mask.shape[1] * scale)
#         from PIL import Image
#         mask_vis = Image.fromarray((mask * 255).astype(np.uint8), mode='L')
#         mask_vis = mask_vis.resize((new_w, new_h), Image.Resampling.NEAREST)
#         mask_vis = np.array(mask_vis) / 255.0
#     else:
#         mask_vis = mask
#     
#     axes[region_idx].imshow(mask_vis, cmap='gray')
#     axes[region_idx].set_title(f"Região {region_idx} - Máscara Modificada\n{Path(args['ortho_images'][region_idx]).name}")
#     axes[region_idx].axis('off')
# 
# plt.tight_layout()
# plt.show()

## 8. Salvar Máscaras como TIFF

In [ ]:
# Determinar pasta de saída para máscaras geradas
data_path = args.get('data_path', '.')
masks_folder = Path(data_path) / "generated_masks"
check_folder(str(masks_folder))

print(f"Pasta de saída: {masks_folder}")

# Salvar máscaras
saved_mask_paths = {}

for region_idx in range(num_regions):
    mask = masks[region_idx]
    metadata = orthoimages_metadata[region_idx]
    ortho_path = args['ortho_images'][region_idx]
    
    # Gerar nome do arquivo baseado no nome da orthoimage
    from os.path import basename, splitext
    ortho_name = splitext(basename(ortho_path))[0]
    output_path = masks_folder / f"{ortho_name}_mask.tif"
    
    print(f"\nSalvando máscara região {region_idx}...")
    print(f"  Arquivo: {output_path}")
    print(f"  Shape: {mask.shape}")
    print(f"  Dtype: {mask.dtype}")
    print(f"  Pixels válidos: {np.sum(mask):,} / {mask.size:,} ({100*np.sum(mask)/mask.size:.2f}%)")
    
    # Salvar TIFF
    array2raster(
        str(output_path), 
        mask, 
        metadata, 
        dtype='uint8'
    )
    
    saved_mask_paths[region_idx] = str(output_path)
    print(f"  ✓ Máscara salva com sucesso!")
    
    # Salvar preview PNG (opcional)
    if save_preview:
        png_path = output_path.with_suffix('.png').with_name(f"{output_path.stem}_preview.png")
        
        # Converter máscara para 0-255 para visualização
        mask_vis = (mask * 255).astype(np.uint8)
        
        # Criar imagem PIL
        img = Image.fromarray(mask_vis, mode='L')
        
        # Calcular dimensões de redimensionamento mantendo aspect ratio
        width, height = img.size
        if width > height:
            if width > preview_max_size:
                new_width = preview_max_size
                new_height = int(height * preview_max_size / width)
            else:
                new_width, new_height = width, height
        else:
            if height > preview_max_size:
                new_height = preview_max_size
                new_width = int(width * preview_max_size / height)
            else:
                new_width, new_height = width, height
        
        # Redimensionar se necessário
        if (new_width, new_height) != (width, height):
            img = img.resize((new_width, new_height), Image.Resampling.NEAREST)
        
        # Salvar PNG
        img.save(str(png_path), optimize=True)
        print(f"  ✓ Preview PNG salvo: {png_path.name} (tamanho: {new_width}x{new_height})")

print("\n" + "="*60)
print("TODAS AS MÁSCARAS FORAM SALVAS COM SUCESSO!")
print("="*60)
for region_idx, path in saved_mask_paths.items():
    print(f"Região {region_idx}: {path}")